In [1]:
!pip install -q transformers torch langdetect


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\furyd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Importing the necessary libraries
import re
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langdetect import detect

C:\Users\furyd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize Translation class
class NotebookTranslator:
    def __init__(self):
        self.model_name = "facebook/nllb-200-distilled-600M"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
        self.LANG_MAP = {
            "en": "eng_Latn", "fr": "fra_Latn", "es": "spa_Latn",
            "de": "deu_Latn", "da": "dan_Latn", "zh": "zho_Hans", "auto": "eng_Latn" 
        }

    def detect_language(self, text):
        try:
            if not text or not any(char.isalpha() for char in text):
                return "en"
            return detect(text)
        except Exception:
            return "en" 

    def translate(self, text, source='auto', target='en'):
        src_lang_code = self.LANG_MAP.get(source, "eng_Latn") 
        tgt_lang_code = self.LANG_MAP.get(target, "eng_Latn") 
        self.tokenizer.src_lang = src_lang_code
        inputs = self.tokenizer(text, return_tensors="pt")
        tgt_lang_id = self.tokenizer.convert_tokens_to_ids(tgt_lang_code)
        
        translated_tokens = self.model.generate(
            **inputs, forced_bos_token_id=tgt_lang_id, max_length=250
        )
        return self.tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

# Instantiate engines
translator = NotebookTranslator()
generator = pipeline("text-generation", model="facebook/blenderbot-400M-distill")

Device set to use cpu


In [4]:
def extract_questions(text):
    return re.findall(r'\d+\..*?\?', text) or [text]

def generate_ai_response(prompt, max_new_tokens=120):
    # Blenderbot works best with text-generation settings tuned down
    res = generator(prompt, max_new_tokens=max_new_tokens, temperature=0.7)
    return res[0]['generated_text']

In [5]:
print("\n" + "="*40)
enquiry_text = input("Enter your enquiry: ").strip()  # User manually inputs enquiry
print("="*40)

In [6]:
# DETECT SOURCE LANGUAGE 
detect_lang = translator.detect_language(enquiry_text)
translated_enq = translator.translate(enquiry_text, source=detect_lang, target='en')

# Print results
print(f"Original Enquiry: {enquiry_text}")
print(f"Detected Language: {detect_lang.upper()}")
print(f"Translated to English (for AI processing): {translated_enq}")

Original Enquiry: Hi! My name is Daniel, and i was wondering what the capital city of Angola was...
Detected Language: EN
Translated to English (for AI processing): My name is Daniel, and I was wondering what the capital of Angola was...


In [7]:
# QUESTION SEARCH
def contains_question(text):
    return '?' in text.strip()

print(f"Contains Question? {contains_question(enquiry_text)}")

Contains Question? False


In [ ]:
# Generate and display responses

# 1. Generate the response in English
ai_response = generate_ai_response(translated_enq, max_new_tokens=350)
print(f"AI Response in English: {ai_response}\n")

# 2. Translate the English response back to the user's native tongue
response_transl = translator.translate(ai_response, source='en', target=detect_lang)
print(f"AI Response in Original Language ({detect_lang.upper()}): {response_transl}")

AI Response in English: My name is Daniel, and I was wondering what the capital of Angola was... I guess.

